# VectrixDB Tiers & Modes

Understanding the different storage tiers and search modes available in VectrixDB.

## 4-Tier System

VectrixDB offers 4 search modes with increasing quality:

| Mode | Components | Use Case |
|------|------------|----------|
| **dense** | Vector similarity | Fast semantic search |
| **hybrid** | Dense + Sparse + Reranker | Better keyword matching |
| **ultimate** | Dense + Sparse + Reranker + ColBERT | Maximum accuracy |
| **graph** | Ultimate + Knowledge Graph | Complex reasoning (GraphRAG) |

### Model Selection

You can now select which models to use for each component:

```python
db = Vectrix(
    "docs",
    mode="ultimate",                    # Default search mode
    dense_model="bge-small",            # Bundled or HuggingFace
    sparse_model="splade",              # Bundled: bm25, splade
    reranker_model="L6",                # Bundled: L6, L12
    late_interaction_model="colbert",   # Bundled: colbert
)
```

In [1]:
from vectrixdb import Vectrix

# Sample data
texts = [
    "Machine learning is a subset of artificial intelligence.",
    "Deep learning uses neural networks with many layers.",
    "Natural language processing handles text and speech.",
    "Computer vision enables machines to interpret images.",
    "Reinforcement learning trains agents through rewards."
]

### Dense Mode
Fastest - uses only vector embeddings (semantic similarity).

In [ ]:
# Dense mode - fastest, semantic only
db_dense = Vectrix("test_dense", mode="dense", dense_model="bge-small")
db_dense.add(texts)

results = db_dense.search("AI and machine learning")
print("Dense mode results:")
for r in results:
    print(f"  {r.score:.4f}: {r.text[:50]}...")

### Hybrid Mode
Combines dense vectors + sparse keywords + cross-encoder reranking.

In [ ]:
# Hybrid mode - dense + sparse + reranker
db_hybrid = Vectrix(
    "test_hybrid",
    mode="hybrid",
    dense_model="bge-small",
    sparse_model="splade",      # Neural sparse (SPLADE++)
    reranker_model="L6",        # Cross-encoder reranker
)
db_hybrid.add(texts)

results = db_hybrid.search("neural networks")
print("Hybrid mode results:")
for r in results:
    print(f"  {r.score:.4f}: {r.text[:50]}...")

### Ultimate Mode
Maximum accuracy: dense + sparse + reranker + ColBERT late interaction.

In [ ]:
# Ultimate mode - dense + sparse + reranker + ColBERT
db_ultimate = Vectrix(
    "test_ultimate",
    mode="ultimate",
    dense_model="bge-small",
    sparse_model="splade",
    reranker_model="L6",
    late_interaction_model="colbert",  # ColBERT for MaxSim scoring
)
db_ultimate.add(texts)

results = db_ultimate.search("text processing")
print("Ultimate mode results:")
for r in results:
    print(f"  {r.score:.4f}: {r.text[:50]}...")

## Model Selection Options

### Bundled Models (Offline, No Download)

| Component | Aliases | Model |
|-----------|---------|-------|
| Dense | `"bge-small"`, `"e5-small"` | BAAI/bge-small-en-v1.5, intfloat/e5-small-v2 |
| Sparse | `"bm25"`, `"splade"` | BM25, SPLADE++ |
| Reranker | `"L6"`, `"L12"` | ms-marco-MiniLM-L6/L12-v2 |
| Late Interaction | `"colbert"` | answerai-colbert-small-v1 |

### HuggingFace Models (Download on First Use)

```python
db = Vectrix(
    "docs",
    mode="hybrid",
    dense_model="BAAI/bge-large-en-v1.5",
    sparse_model="naver/splade-cocondenser-ensembledistil",
    reranker_model="cross-encoder/ms-marco-MiniLM-L-12-v2",
)
```

### GitHub Releases (Download on First Use)

```python
db = Vectrix(
    "docs",
    mode="ultimate",
    late_interaction_model="github:bge-m3",  # BGE-M3 (100+ languages)
    reranker_model="github:reranker-multi",  # Multilingual reranker
)
```

In [5]:
# Compare modes on hybrid tier
query = "learning algorithms"

print(f"Query: '{query}'\n")

# Dense mode
results_dense = db_hybrid.search(query, mode="dense")
print("Dense mode:")
print(f"  Top: {results_dense.top.text[:60]}...\n")

# Sparse mode
results_sparse = db_hybrid.search(query, mode="sparse")
print("Sparse mode:")
print(f"  Top: {results_sparse.top.text[:60]}...\n")

# Hybrid mode
results_hybrid = db_hybrid.search(query, mode="hybrid")
print("Hybrid mode:")
print(f"  Top: {results_hybrid.top.text[:60]}...")

Query: 'learning algorithms'



Dense mode:
  Top: Deep learning uses neural networks with many layers....



Sparse mode:
  Top: Reinforcement learning trains agents through rewards....



Hybrid mode:
  Top: Reinforcement learning trains agents through rewards....


## Reranking

Use `rerank="cross-encoder"` for higher quality results.

In [6]:
# Rerank uses a cross-encoder to reorder results
results_rerank = db_hybrid.search("image recognition", rerank="cross-encoder")

print("Reranked results:")
for r in results_rerank:
    print(f"  {r.score:.4f}: {r.text[:50]}...")

Reranked results:
  -9.1190: Reinforcement learning trains agents through rewar...
  -9.1190: Computer vision enables machines to interpret imag...
  -9.1190: Deep learning uses neural networks with many layer...
  -9.1190: Natural language processing handles text and speec...
  -9.1190: Machine learning is a subset of artificial intelli...


## Cleanup

In [7]:
import shutil, os

for folder in ["test_dense", "test_hybrid", "test_ultimate"]:
    if os.path.exists(folder):
        shutil.rmtree(folder)
        print(f"Deleted {folder}")